# Board Visualization Library Tour
This notebook provides a visual walkthrough of the board rendering system.

The goal is to verify that:

- board states convert correctly into renderable grids
- backgrounds render correctly
- geometry places pieces correctly
- scenes provide a simple high-level API
- chess and Go boards both render consistently

This notebook complements the automated test suite by allowing manual inspection of rendered output.

## Setup
First let's get some imports:

In [ ]:
if "google.colab" in str(get_ipython()):
  !pip install git+https://github.com/DiogenesAnalytics/games

In [ ]:
# chess/go engines
import chess
from sgfmill import boards

# visualization scene classes
from games.visualization.board.scene.chess import ChessScene
from games.visualization.board.scene.go import GoScene

## Chess Rendering
Next let's begin with rendering a chess board state.

### Initial Chess Position

Render the standard starting position.

In [ ]:
board = chess.Board()
scene = ChessScene(board)
scene.render()

### Midgame Position
Apply a few moves and confirm piece placement updates correctly:

In [ ]:
board = chess.Board()
board.push_san("e4")
board.push_san("e5")
board.push_san("Nf3")
board.push_san("Nc6")
board.push_san("Bb5")

scene = ChessScene(board)
scene.render()

### Empty Chess Board
Useful for verifying coordinate alignment independently of pieces.

In [ ]:
board = chess.Board(None)
scene = ChessScene(board)
scene.render()

### Custom Chess Position
Render a manually constructed position:

In [ ]:
board = chess.Board(None)
board.set_piece_at(chess.E4, chess.Piece(chess.KING, chess.WHITE))
board.set_piece_at(chess.D5, chess.Piece(chess.KING, chess.BLACK))
board.set_piece_at(chess.C3, chess.Piece(chess.QUEEN, chess.WHITE))

scene = ChessScene(board)
scene.render()

## FEN Chess Position
Setup the board with a [FEN](https://en.wikipedia.org/wiki/Forsyth%E2%80%93Edwards_Notation) position:

In [ ]:
board = chess.Board("r1bqkb1r/pppp1Qpp/2n2n2/4p3/2B1P3/8/PPPP1PPP/RNB1K1NR b KQkq - 0 4")
scene = ChessScene(board)
scene.render()

## Go Rendering
Now let's move on to trying to render a go board state

### Empty 9x9 Board
Verify board centering, grid lines, and star points:

In [ ]:
board = boards.Board(9)
scene = GoScene(board)
scene.render()

### Single Stone
Verify a single stone appears at the correct intersection:

In [ ]:
board = boards.Board(9)
board.play(4, 4, "b")

scene = GoScene(board)
scene.render()

### Small Position
Render multiple stones with both colors.

In [ ]:
board = boards.Board(9)
board.play(2, 2, "b")
board.play(2, 3, "w")
board.play(3, 2, "b")
board.play(3, 3, "w")

scene = GoScene(board)
scene.render()

### Larger 19x19 Board
Verify scaling remains correct on standard board size:

In [ ]:
board = boards.Board(19)
board.play(3, 3, "b")
board.play(15, 15, "w")
board.play(9, 9, "b")

scene = GoScene(board)
scene.render()

## Rendering Consistency Checks

The following visual checks are useful:

### Chess
Confirm:
- pieces are centered in squares
- board colors alternate correctly
- no overlapping pieces
- orientation matches expected coordinates

### Go
Confirm:
- grid is centered
- star points are correctly placed
- stones sit on intersections
- board margins are symmetric

## Low-Level Renderer Access
Advanced users may inspect the underlying matplotlib axis.

In [ ]:
board = chess.Board()
scene = ChessScene(board)

ax = scene.render(return_ax=True)

print("Rendered text objects:", len(ax.texts))
print("Axis limits:", ax.get_xlim(), ax.get_ylim())

## Rendering Configuration
There is also an interface for configuring the rendering:

In [ ]:
from games.visualization.board.renderer import RenderSpec

fig_count = 1

# use FEN to setup board
board = chess.Board("r2q1rk1/ppp2ppp/2npbn2/3Np3/2P1P3/2N1B3/PP3PPP/R2QKB1R w KQ - 0 12")

# configure scene for rendering
scene = ChessScene(
    board=board,
    spec=RenderSpec(
        dpi=120,
        subtitle=f"Figure {fig_count}. Chaotic midgame render test",
        subtitle_y=0.08
    )
)

# render
scene.render()

# +1
fig_count+=1

In [ ]:
from sgfmill import sgf

# sgf bytes defining go board state
sgf_bytes = b"""
(;FF[4]GM[1]SZ[19]

AB
[dd][dp][pd][pp][jj][jd][dj][pq][qp][cq][qc][cd][dc][ce][ec][cf][fc][cg][gc][ch][hc]
[bb][bc][cb][cc][bd][db][be][eb][bf][fb][bg][gb]
[de][ed][df][fd][dg][gd][dh][hd][di][id]
[ee][ef][fe][ff][eg][ge][eh][he][ei][ie][ej][je][ek][ke]
[fj][jf][fk][kf][fl][lf][fm][mf]
[gg][gh][hg][hi][ih][hj][jh][hk][kh]
[ij][ji][ik][ki][il][li][im][mi]
[jo][oj][jp][pj][jq][qj]

AW
[ce][ec][cf][fc][cg][gc][ch][hc][ci][ic][cj][jc]
[dd][dp][pd][pp][jj][jd][dj][pq][qp][cq][qc]
[ee][ed][de][df][fd][eg][ge][eh][he][ei][ie][ej][je]
[pf][fp][pg][gp][ph][hp][pi][ip]
[bb][bc][cb][bd][db][be][eb][bf][fb][bg][gb]
[gg][gh][hg][hi][ih][hj][jh][hk][kh]
[kk][kl][lk][ll][lm][ml][mn][nm]
)
"""

# load state and create board
game = sgf.Sgf_game.from_bytes(sgf_bytes)
board = boards.Board(game.get_size())

# now place pieces
root = game.get_root()
black, white, _ = root.get_setup_stones()

for r, c in black:
    board.board[r][c] = 'b'

for r, c in white:
    board.board[r][c] = 'w'

# now get go scene configured
scene = GoScene(
    board=board,
    spec=RenderSpec(
        dpi=120,
        subtitle=f"Figure {fig_count}. Dense go board state",
        subtitle_y=0.08
    )
)

# render
scene.render()

# +1
fig_count+=1

## Summary
This notebook demonstrates the full rendering pipeline:

board state  
→ adapter  
→ grid  
→ geometry  
→ background  
→ renderer  
→ matplotlib output

It is intended as a manual companion to the automated test suite.